### Cell 1 — create Gold schema + generate dim_date

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS mia_catalog.gold")

from pyspark.sql.functions import *

date_range_df = spark.sql("""
    SELECT explode(sequence(to_date('2024-01-01'), to_date('2027-12-31'), interval 1 day)) AS full_date
""")

dim_date_df = date_range_df.select(
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_sk"),
    col("full_date"),
    year(col("full_date")).alias("year"),
    quarter(col("full_date")).alias("quarter"),
    month(col("full_date")).alias("month"),
    dayofmonth(col("full_date")).alias("day"),
    dayofweek(col("full_date")).alias("day_of_week"),
    date_format(col("full_date"), "EEEE").alias("day_name"),
    date_format(col("full_date"), "MMMM").alias("month_name"),
)

dim_date_df.write.format("delta").mode("overwrite").saveAsTable("mia_catalog.gold.dim_date")
print(f"dim_date created: {dim_date_df.count()} rows")

dim_date created: 1461 rows
